# 用 Sciverse 下载论文图表资源

> 从全文 Markdown 中提取图表路径，通过 resource 接口获取二进制文件

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 读取全文并提取图表路径

从 content 接口返回的 Markdown 中提取图片引用


In [ ]:
import httpx, re

BASE = "https://api.sciverse.space"
HEADERS = {"Authorization": "Bearer sv-..."}

async def get_content(doc_id: str):
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"{BASE}/content",
            headers=HEADERS,
            params={"doc_id": doc_id, "offset": 0, "limit": 8192}
        )
        return resp.json()["content"]

content = await get_content("af2_nature_2021")
# 提取 Markdown 图片路径
figure_paths = re.findall(r'!\[.*?\]\((.*?)\)', content)
print(f"Found {len(figure_paths)} figures")
for p in figure_paths:
    print(f"  - {p}")

## Step 2: 下载图表文件

通过 resource 接口获取图片二进制数据


In [ ]:
from pathlib import Path

async def download_resource(path: str, save_dir: str = "./figures"):
    Path(save_dir).mkdir(exist_ok=True)
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"{BASE}/resource",
            headers=HEADERS,
            params={"path": path}
        )
        filename = path.split("/")[-1]
        local_path = f"{save_dir}/{filename}"
        Path(local_path).write_bytes(resp.content)
        return local_path

# 下载所有图表
for fig_path in figure_paths[:5]:
    local = await download_resource(fig_path)
    print(f"Downloaded: {local}")

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
